# Chapter 5 - Question 6
### Group 4

We use logistic regression to predict `default` from `income` and `balance` on the `Default` dataset. We find the standard errors of the coefficients two ways: with the bootstrap, and with the formula sm.GLM() gives us automatically.

## Load the data

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from ISLP import load_data
from ISLP.models import ModelSpec as MS, summarize

np.random.seed(1)  # so results are reproducible

Default = load_data('Default')
Default.head()


## (a) Standard errors from sm.GLM()

We fit the logistic regression with income and balance as predictors and look at the std err column.

In [ ]:
X = MS(['income', 'balance']).fit_transform(Default)
y = (Default['default'] == 'Yes').astype(float)

glm_model = sm.GLM(y, X, family=sm.families.Binomial())
glm_results = glm_model.fit()
summarize(glm_results)


## (b) boot_fn()

Takes the data and a set of row indices, refits the model on just those rows, and gives back the income and balance coefficients.

In [ ]:
def boot_fn(data, idx):
    D = data.iloc[idx]
    X_ = MS(['income', 'balance']).fit_transform(D)
    y_ = (D['default'] == 'Yes').astype(float)
    results_ = sm.GLM(y_, X_, family=sm.families.Binomial()).fit()
    return results_.params[['income', 'balance']]

# check it matches part (a) when we use all the rows
boot_fn(Default, np.arange(Default.shape[0]))


## (c) Bootstrap standard errors

We take 1000 bootstrap samples (same size as the data, sampled with replacement), run boot_fn() on each one, and look at how much the coefficients move around.

In [ ]:
n = Default.shape[0]
B = 1000

boot_coefs = np.zeros((B, 2))

for b in range(B):
    idx = np.random.choice(n, size=n, replace=True)
    coefs = boot_fn(Default, idx)
    boot_coefs[b, 0] = coefs['income']
    boot_coefs[b, 1] = coefs['balance']

boot_se_income = boot_coefs[:, 0].std()
boot_se_balance = boot_coefs[:, 1].std()

print("Bootstrap SE (income):", boot_se_income)
print("Bootstrap SE (balance):", boot_se_balance)


## (d) Comparing the two standard errors

In [ ]:
comparison = pd.DataFrame({
    'GLM formula SE': [glm_results.bse['income'], glm_results.bse['balance']],
    'Bootstrap SE': [boot_se_income, boot_se_balance]
}, index=['income', 'balance'])

comparison


**Comment:**

The two sets of standard errors are pretty close to each other for both income and balance. This makes sense because the sm.GLM() formula is based on large-sample theory, and with n = 10000 observations we have plenty of data for that to hold up well.

The main difference is that the bootstrap doesn't assume anything about the shape of the sampling distribution - it just resamples the data over and over and measures how much the coefficients change. The formula method is faster since it's just one calculation, but the bootstrap is more trustworthy when the sample is small or the model's assumptions don't really hold. Here they agree closely, so it's a good sign that both methods are giving us a reliable estimate.